In [26]:
import pathlib
import numpy as np 
import torch
import open3d as o3d

path_to_models = pathlib.Path("/home/yefim-home/Downloads/Telegram Desktop/dataset/power_drills")
directories = [dir.name for dir in path_to_models.iterdir() if dir.is_dir()]

mesh_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
            size=0.1, origin=[0, 0, 0])
for dir in directories[:1]:
    path_to_pc = path_to_models / dir / "point_cloud_labeled.ply"
    path_to_mesh = path_to_models / dir / "object_convex_decomposition_meter_unit.obj"
    mesh = o3d.io.read_triangle_mesh(path_to_mesh)
    o3d_pcd = o3d.io.read_point_cloud(path_to_pc).scale(0.01, np.array([0.0,0.0,0.0]))
    o3d.visualization.draw_geometries([mesh_frame, o3d_pcd])

In [41]:
frame_pos = torch.randn(9).reshape(3,3) * 1
frame_pos

tensor([[ 0.7441,  0.3150, -0.4556],
        [ 0.7826,  1.0276,  0.7738],
        [ 0.2082, -0.1215,  1.3196]])

In [42]:
pcd = torch.from_numpy(np.asarray(o3d_pcd.points))

indeces = []
vectors2closest_point = []
for frm_pos in frame_pos:
    vectors = frm_pos - pcd
    min_norm_id = torch.linalg.norm(vectors, dim=1).argmin()
    indeces.append(min_norm_id)
    vectors2closest_point.append(vectors[min_norm_id])
    print(min_norm_id)
    print(vectors[min_norm_id])
    
indeces = torch.tensor(indeces, dtype=torch.int64)
vectors2closest_point = torch.stack(vectors2closest_point)
pcd[indeces], vectors2closest_point

tensor(4713)
tensor([-0.0265, -0.3043, -0.3150], dtype=torch.float64)
tensor(1379)
tensor([0.1227, 0.1528, 0.5272], dtype=torch.float64)
tensor(310)
tensor([0.0864, 0.4965, 0.9297], dtype=torch.float64)


(tensor([[ 0.7707,  0.6193, -0.1406],
         [ 0.6598,  0.8749,  0.2466],
         [ 0.1218, -0.6180,  0.3899]], dtype=torch.float64),
 tensor([[-0.0265, -0.3043, -0.3150],
         [ 0.1227,  0.1528,  0.5272],
         [ 0.0864,  0.4965,  0.9297]], dtype=torch.float64))

In [43]:
mesh_frames = []
line_points = []
lines = []
for k, id in enumerate(indeces.tolist()):
    o3d_pcd.colors[id] = np.array([0.0, 1.0, 0.0])
    mesh_frames.append(o3d.geometry.TriangleMesh.create_coordinate_frame(
                size=0.5, origin=frame_pos[k].numpy()))
    
    line_points.append(pcd[id].tolist())
    line_points.append(frame_pos[k].tolist())
    lines.append([len(line_points)-2, len(line_points)-1])
    
colors = [[0, 0, 1] for i in range(len(lines))]
line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(line_points),
    lines=o3d.utility.Vector2iVector(lines),
)
line_set.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries(mesh_frames + [o3d_pcd] + [line_set])

In [24]:
mesh_frames

[TriangleMesh with 1134 points and 2240 triangles.,
 TriangleMesh with 1134 points and 2240 triangles.,
 TriangleMesh with 1134 points and 2240 triangles.]

torch.Size([3, 3])

In [52]:
vectors = pcd.unsqueeze(0).expand(frame_pos.shape[0], -1, -1) - frame_pos.unsqueeze(1)

min_indeces = torch.linalg.norm(vectors, dim=2).argmin(1)
min_indeces

vectors[list(range(frame_pos.shape[0])), min_indeces, :]

tensor([[-1.0385, -0.4420, -4.3833],
        [-1.9180, -1.6247, -1.6935],
        [-0.0631,  1.4288,  0.0487]], dtype=torch.float64)